# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # `metadata` is an object, do not subscript

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available RecordSets in the dataset, their @id and fields
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")
for i, record_set in enumerate(record_sets):
    print(f"Record Set {i+1} - Name: {getattr(record_set, 'name', None)} | @id: {getattr(record_set, '@id', None)}")
    print("  Fields (@id):")
    for field in record_set.fields:
        print(f"    - {getattr(field, 'name', None)} (@id: {getattr(field, '@id', None)}) | dataType: {getattr(field, 'data_type', None)}")
    print("---")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll extract all available record sets into DataFrames for easy access
dataframes = {}
record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None) for rs in record_sets]
print("Record set @ids:", record_set_ids)

for record_set in record_sets:
    record_set_id = getattr(record_set, '@id', None)
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from RecordSet: {record_set_id}")
        print(f"Fields: {list(df.columns)}\n")
    except Exception as e:
        print(f"Could not load records for RecordSet {record_set_id}: {e}")

# For illustration: Show the first dataframe's columns and a preview
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Columns for {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a main record set and fields for EDA
# Edit these as needed based on previous output
main_record_set_id = record_set_ids[0]  # Use the first RecordSet by default
main_df = dataframes[main_record_set_id].copy()

# Find a numeric field (e.g., 'MW' for molecular weight, or other numeric columns)
# Let's auto-detect a likely numeric field
numeric_fields = main_df.select_dtypes(include=['number']).columns.tolist()
if not numeric_fields:
    # Attempt to convert columns to numeric if possible
    for col in main_df.columns:
        main_df[col+'_num'] = pd.to_numeric(main_df[col], errors='coerce')
    numeric_fields = main_df.select_dtypes(include=['number']).columns.tolist()

print(f"Numeric fields detected: {numeric_fields}")
if numeric_fields:
    numeric_field = numeric_fields[0]  # Take the first numeric field
+    print(f"Using numeric field '{numeric_field}' for filtering and normalization.")
    threshold = main_df[numeric_field].mean()  # Use mean as a threshold example
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")

    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a string/categorical field if available
    candidate_group_fields = main_df.select_dtypes(include=['object']).columns.tolist()
    group_field = None
    for field in candidate_group_fields:
        n_unique = main_df[field].nunique(dropna=True)
        if n_unique > 1 and n_unique < (0.7 * len(main_df)):
            group_field = field
            break
    if group_field:
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
        print(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group_field
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print("No numeric field to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded and explored the [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using `mlcroissant`.

- We inspected all record sets and relevant fields using their `@id`s for robust referencing.
- We demonstrated data extraction into Pandas DataFrames for programmatic use.
- Exploratory data analysis was performed by filtering, normalizing, and grouping numeric fields.
- Visualizations revealed data distributions, and relationships between selected attributes.

This workflow can be adapted to other Croissant-structured datasets by adjusting record set and field `@id`s as revealed during exploration.